# Assignment 1B — Domain LLM Adaptation & Production Optimization
**Domain:** HR Policy & Corporate Documents  |  **Model:** `HuggingFaceTB/SmolLM2-1.7B-Instruct` (same tokenizer throughout)  |  **Draft model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  |  **GPU:** Colab T4

**Group 53 — BITS ID, Name, % contribution** (fill in all four before submitting)

| # | BITS ID | Name | % contribution |
|---|---------|------|----------------|
| 1 | 2025AF05052 | ASHA RANI | __ % |
| 2 | 2025AF05031 | JOSHI SHRIKANT SHRIDHARRA | __ % |
| 3 | 2025AE05319 | KOLLI LIKHITHA | __ % |
| 4 | 2025AF05040 | SHIROLKAR NEEHALI SUNIL | __ % |

**Pipeline:** PDFs → cleaned `domain_corpus/*.txt` → grounded instruction dataset → QLoRA adapter → decoding comparison → speculative decoding → 4-bit quantization → cost per 1M tokens.

> **Run order:** Runtime → *Change runtime type* → **T4 GPU** → run the install cell → **Runtime → Restart session** → run everything top to bottom (never skip a cell after a restart; every cell below is restart-safe except that Part A Step 1 must run before Part B).
> All numbers, tables and inferences below are produced by code, so the narrative always matches the measurements.

**T4 note (precision).** The T4 (compute capability 7.5) has no native bfloat16 tensor cores. The notebook therefore uses **float16** on the T4 for every step the assignment words as "bfloat16" (same 2 bytes/parameter, same memory), and switches to true **bfloat16** automatically on A100/L40S/L4. All tables print the dtype actually used. Set `FORCE_BF16 = True` in the setup cell to force bfloat16 on a T4 (works, but emulated and several times slower).

In [ ]:
# Run once, then restart the runtime (Runtime -> Restart session) and continue from the next cell.
!pip install -q transformers==4.46.3 datasets==2.21.0 huggingface_hub==0.24.7 peft==0.13.2 trl==0.11.4 accelerate==1.0.1 bitsandbytes pypdf langdetect sentencepiece pandas numpy==1.26.4 scipy==1.13.1

In [ ]:
import os, re, glob, json, gc, time, random, hashlib, shutil, csv, math
import numpy as np, pandas as pd, torch
import transformers, peft, trl, datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

assert torch.cuda.is_available(), "GPU required: Runtime -> Change runtime type -> T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
# Assignment rates: Rs 3.50/hr for T4, Rs 12.00/hr for A100 (L40S treated like A100-class)
HOURLY_RATE = 3.50 if "T4" in GPU_NAME else 12.00

# ---- precision: bfloat16 where the GPU supports it natively, float16 on T4 ----
FORCE_BF16 = False
NATIVE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if (NATIVE_BF16 or FORCE_BF16) else torch.float16
DTYPE_NAME = "bfloat16" if DTYPE is torch.bfloat16 else "float16"
USE_BF16_TRAIN = DTYPE is torch.bfloat16

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
DRAFT_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
CORPUS_DIR = "domain_corpus"

PDF_DIR = None
for cand in ["datavaol", "/content/datavaol", "/content/drive/MyDrive/datavaol", "/content"]:
    if len(glob.glob(f"{cand}/*.pdf")) >= 1:
        PDF_DIR = cand; break
if PDF_DIR is None:
    raise FileNotFoundError("Upload the HR PDFs (>=5) into /content/datavaol/ (or mount Drive) and re-run this cell.")

print("Torch", torch.__version__, "| Transformers", transformers.__version__, "| PEFT", peft.__version__, "| TRL", trl.__version__)
print(f"GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB) | native bf16: {NATIVE_BF16} | working dtype: {DTYPE_NAME}")
print("hourly rate used for cost formula: Rs", HOURLY_RATE)
print("PDF dir:", PDF_DIR, "|", len(glob.glob(f"{PDF_DIR}/*.pdf")), "PDFs")

In [ ]:
# ---- Shared helpers & FIXED prompts (used unchanged in Parts A, B3 and C) ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

PROMPTS = [
    "Explain the leave types available for employees and how leave applications are handled.",
    "What guidelines should hiring managers follow when recruiting and onboarding employees?",
    "Explain the company's hybrid and remote work policy and employee responsibilities.",
]

def build_prompt(user_msg):
    """SmolLM2 ChatML format WITHOUT a system prompt (the stock template would inject a default one)."""
    return f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"

def build_sft_text(instr, resp):
    return build_prompt(instr) + f"{resp}<|im_end|>\n"

def free():
    gc.collect(); torch.cuda.empty_cache()

def safe_del(*names):
    """Delete globals if they exist (keeps cells re-runnable after a restart / partial run)."""
    for n in names:
        globals().pop(n, None)
    free()

@torch.no_grad()
def generate(model, prompt, cfg, max_new_tokens=150, assistant_model=None, seed=SEED):
    """Returns (text of NEW tokens only, seconds, n_new_tokens)."""
    enc = tokenizer(build_prompt(prompt), return_tensors="pt").to(model.device)
    set_seed(seed)
    kw = dict(cfg)
    if assistant_model is not None:
        kw["assistant_model"] = assistant_model
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=IM_END_ID, **kw)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    new = out[0, enc["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip(), dt, int(new.shape[0])

# ---- Output-quality proxies (computed, not eyeballed) ----
_CORPUS_TRI = None
def _words(t): return re.findall(r"[a-z0-9']+", t.lower())
def corpus_trigrams():
    global _CORPUS_TRI
    if _CORPUS_TRI is None:
        s = set()
        for f in glob.glob(f"{CORPUS_DIR}/*.txt"):
            w = _words(open(f, encoding="utf-8").read())
            s.update(" ".join(w[i:i+3]) for i in range(len(w) - 2))
        _CORPUS_TRI = s
    return _CORPUS_TRI

ARTIFACT_RE = re.compile(r"reword|rewrite|how can this be improved|key details:|analysis:|^question:|^answer:", re.I | re.M)
def metrics(text):
    w = _words(text)
    tri = [" ".join(w[i:i+3]) for i in range(len(w) - 2)]
    bi = [(w[i], w[i+1]) for i in range(len(w) - 1)]
    ctri = corpus_trigrams()
    return dict(words=len(w),
                corpus_trigram_overlap=round(sum(t in ctri for t in tri) / max(1, len(tri)), 3),
                distinct_2=round(len(set(bi)) / max(1, len(bi)), 3),
                artifact=bool(ARTIFACT_RE.search(text)))
print("Fixed prompts:"); [print(" -", p) for p in PROMPTS]

# PART A — Domain Data Collection & Cleaning  [2 marks]

## Step 1 — Corpus building and cleaning
Design decisions (each justified):
* **Extraction:** `pypdf`, page by page, so page-level filters/dedup are possible.
* **Length filter:** (a) pages under **100 characters** are dropped (blank, cover, image-only or heading-only pages carry no policy content); (b) documents under **5,000 characters (~800–1,000 words)** after that are dropped, because the instruction-dataset step needs several 120–200-word chunks per document.
* **Deduplication:** (a) exact duplicate documents via **MD5 of normalised text**; (b) repeated **pages/sections** via exact page hash plus **near-duplicate detection** (Jaccard similarity of word 5-gram shingles ≥ 0.85) across the whole corpus.
* **Language filter:** `langdetect` (seeded) on up to 20 samples of 1,000 characters per document; keep a document if ≥ 80 % of samples are English.
* Statistics (PDFs, .txt files, pages, characters) are recorded after every step.

In [ ]:
from pypdf import PdfReader

pdf_files = sorted(glob.glob(f"{PDF_DIR}/*.pdf"))
assert len(pdf_files) >= 5, "need at least 5 PDFs"
docs = {}                      # name -> list of page texts
for p in pdf_files:
    pages = [(pg.extract_text() or "").replace("\x00", "") for pg in PdfReader(p).pages]
    docs[os.path.splitext(os.path.basename(p))[0]] = pages

stats_rows = []
def snap(step):
    stats_rows.append(dict(Step=step, PDFs=len(docs), TXT_files=len(docs),
                           Pages=sum(len(v) for v in docs.values()),
                           Chars=sum(len(pg) for v in docs.values() for pg in v)))
snap("0. Raw extraction")

# ---- Length filter ----
MIN_PAGE_CHARS, MIN_DOC_CHARS = 100, 5000
for k in list(docs):
    docs[k] = [pg for pg in docs[k] if len(pg.strip()) >= MIN_PAGE_CHARS]
for k in list(docs):
    if sum(len(pg) for pg in docs[k]) < MIN_DOC_CHARS:
        print("Removed (too short):", k); del docs[k]
snap("1. After length filter")

# ---- Deduplication ----
def norm(t): return re.sub(r"\W+", " ", t.lower()).strip()
def shingles(t, k=5):
    w = norm(t).split()
    return {" ".join(w[i:i+k]) for i in range(max(1, len(w) - k + 1))}
def jaccard(a, b): return len(a & b) / max(1, len(a | b))

seen_docs = {}
for k in list(docs):
    h = hashlib.md5(norm(" ".join(docs[k])).encode()).hexdigest()
    if h in seen_docs:
        print(f"Removed duplicate document: {k} == {seen_docs[h]}"); del docs[k]
    else:
        seen_docs[h] = k

kept_sh, kept_hash, removed_pages = [], set(), 0
for k in list(docs):
    new_pages = []
    for pg in docs[k]:
        h = hashlib.md5(norm(pg).encode()).hexdigest()
        sh = shingles(pg)
        if h in kept_hash or any(jaccard(sh, s) >= 0.85 for s in kept_sh):
            removed_pages += 1; continue
        kept_hash.add(h); kept_sh.append(sh); new_pages.append(pg)
    docs[k] = new_pages
for k in list(docs):
    if sum(len(pg) for pg in docs[k]) < MIN_DOC_CHARS:
        print("Removed (too short after dedup):", k); del docs[k]
print("Duplicate / near-duplicate pages removed:", removed_pages)
snap("2. After deduplication")

# ---- Language filter ----
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def is_english(text, n=20, size=1000, thr=0.8):
    t = re.sub(r"\s+", " ", text)
    chunks = [t[i:i+size] for i in range(0, max(1, len(t) - size), size)] or [t]
    random.Random(SEED).shuffle(chunks)
    votes = []
    for c in chunks[:n]:
        try: votes.append(detect(c) == "en")
        except Exception: votes.append(False)
    return sum(votes) / len(votes) >= thr
for k in list(docs):
    if not is_english("\n".join(docs[k])):
        print("Removed (non-English):", k); del docs[k]
snap("3. After language filter")

# ---- Write cleaned corpus ----
shutil.rmtree(CORPUS_DIR, ignore_errors=True); os.makedirs(CORPUS_DIR)
for k, pages in docs.items():
    with open(os.path.join(CORPUS_DIR, k + ".txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(pages))
snap("4. Final .txt written to domain_corpus/")
stats_df = pd.DataFrame(stats_rows); display(stats_df)
stats_df.to_csv("corpus_statistics.csv", index=False)
assert len(glob.glob(f"{CORPUS_DIR}/*.txt")) >= 5, "fewer than 5 documents survived cleaning - add more PDFs"
print("Files in domain_corpus/:", len(glob.glob(f"{CORPUS_DIR}/*.txt")))

In [ ]:
# Which step reduced the corpus most? (computed) + automatic explanation
d = stats_df.copy()
d["Files_removed"] = -d["TXT_files"].diff().fillna(0).astype(int)
d["Pages_removed"] = -d["Pages"].diff().fillna(0).astype(int)
d["Chars_removed"] = -d["Chars"].diff().fillna(0).astype(int)
display(d[["Step", "Files_removed", "Pages_removed", "Chars_removed"]])
steps = d.iloc[1:4].reset_index(drop=True)
top = steps.loc[steps["Chars_removed"].idxmax()]
WHY = {
 "1. After length filter": "near-empty pages (covers, separators, image-only or heading-only pages) and stub documents carry no policy text, so they are the first thing to go.",
 "2. After deduplication": "HR handbooks share templates, repeated headers/footers, disclaimers and re-issued policy sections, so many pages are exact or near-exact copies of pages in other files.",
 "3. After language filter": "one or more source documents were written in another language, so whole files were dropped at once.",
}
print(f"Largest reduction: '{top['Step']}' -> {int(top['Pages_removed'])} pages, {int(top['Chars_removed'])} characters, {int(top['Files_removed'])} files "
      f"({100*top['Chars_removed']/max(1,d['Chars'].iloc[0]):.1f}% of the raw characters).")
if top["Chars_removed"] == 0:
    print("No step removed anything: the source PDFs were already clean, English and distinct.")
else:
    print("Why:", WHY[top["Step"]])
for i in (1, 2, 3):
    r = d.iloc[i]
    print(f"  {r['Step']}: -{int(r['Files_removed'])} files, -{int(r['Pages_removed'])} pages, -{int(r['Chars_removed'])} chars")
print(f"\nInference: starting from {int(d['PDFs'].iloc[0])} PDFs / {int(d['Pages'].iloc[0])} pages / {int(d['Chars'].iloc[0]):,} characters, the cleaned corpus has "
      f"{int(d['TXT_files'].iloc[-1])} files / {int(d['Pages'].iloc[-1])} pages / {int(d['Chars'].iloc[-1]):,} characters. Removing repeated and near-empty text matters because "
      "boilerplate that appears many times would be over-weighted during fine-tuning and would fill the instruction dataset with near-identical examples.")

## Step 2 — Baseline model output (no system prompt)
The base model is loaded in **bfloat16** (float16 on the T4 — see the note at the top). Greedy decoding is used for the baseline **and** for the adapter in Part B3, so the two columns differ only because of the weights, not sampling noise. Prompts use the model's ChatML format without a system prompt.

In [ ]:
safe_del("base_model")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0}).eval()

total_params = sum(p.numel() for p in base_model.parameters())
arch = {"Model": MODEL_ID, "dtype": str(next(base_model.parameters()).dtype),
        "Total parameters": f"{total_params:,} ({total_params/1e9:.2f}B)",
        "Decoder layers": base_model.config.num_hidden_layers,
        "Hidden size": base_model.config.hidden_size,
        "Vocabulary size": base_model.config.vocab_size,
        "Attention heads": base_model.config.num_attention_heads}
display(pd.DataFrame(arch.items(), columns=["Architecture item", "Value"]))

GREEDY = dict(do_sample=False)
baseline_outputs = []
for p in PROMPTS:
    text, dt, n = generate(base_model, p, GREEDY, max_new_tokens=150)
    baseline_outputs.append({"prompt": p, "response": text})
    print("=" * 90, "\nPROMPT:", p, "\n\nBASELINE RESPONSE:\n", text, "\n")
assert all(len(o["response"]) > 20 for o in baseline_outputs), "empty baseline output - model produced NaN/garbage in this dtype"
pd.DataFrame(baseline_outputs).to_csv("baseline_outputs.csv", index=False)
print("saved baseline_outputs.csv")

In [ ]:
bm = pd.DataFrame([{"Prompt": o["prompt"][:50] + "...", **metrics(o["response"])} for o in baseline_outputs]); display(bm)
print(f"Inference (computed): the baseline answers average {bm.words.mean():.0f} words; only {100*bm.corpus_trigram_overlap.mean():.1f}% of their word-trigrams occur anywhere in the HR corpus, "
      f"i.e. the base model answers with generic HR phrasing rather than the organisation's own policy wording. "
      "It cannot know organisation-specific entitlements, eligibility rules, approval workflow or approvers because they were never in its pre-training data - "
      "the gap that the QLoRA adaptation in Part B tries to close. Part B3 compares against exactly these saved outputs.")

# PART B — Instruction Fine-Tuning with QLoRA  [5 marks]

## Part B1 — Instruction dataset creation (grounded Q&A)  [2 marks]
Every example is a **self-contained question + short answer generated only from a corpus chunk** (raw text slices under a generic instruction would teach nothing about which question maps to which answer).

* **Source:** only the cleaned `domain_corpus/*.txt` files.
* **Chunking:** sentence-aligned chunks of 120–200 words; tables/numeric junk filtered (alphabetic ratio ≥ 0.7); a seeded random sample of 300 chunks.
* **Method:** synthetic generation with an external instruct LLM run locally — `Qwen/Qwen2.5-7B-Instruct` loaded in **4-bit** so it fits the 15 GB T4 (the bf16 weights alone are ~15 GB and run out of memory) — greedy decoding, using the exact template printed below. Falls back to `SmolLM2-1.7B-Instruct` if Qwen cannot be loaded.
* **Quality filters:** valid JSON; question must not refer to "the text/passage"; response 8–120 words; **grounding check** — ≥ 70 % of the response's content words must occur in the source chunk; duplicate questions removed.
* **Size justification:** a few hundred short examples (~100–150 tokens each) are enough for a 1.7B model with a LoRA adapter to pick up terminology while training in minutes on a T4; much more mostly adds repetition from overlapping handbooks.

In [ ]:
PROMPT_TEMPLATE = """Read the text below and generate 3 instruction-response pairs in JSON format based ONLY on this text. Each entry must have "instruction" and "response" keys.
Rules:
- The instruction must be a self-contained question an employee could ask about the HR policy. It must NOT mention "the text", "the passage" or "this section".
- The response must be 1-3 sentences and use only facts stated in the text. Do not add outside knowledge.
Return only a JSON list.

Text:
{chunk}

JSON:"""
print(PROMPT_TEMPLATE)

def make_chunks(text, lo=120, hi=200):
    sents = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text))
    chunks, cur, n = [], [], 0
    for s in sents:
        w = len(s.split())
        if n + w > hi and n >= lo:
            chunks.append(" ".join(cur)); cur, n = [], 0
        cur.append(s); n += w
    if n >= lo: chunks.append(" ".join(cur))
    return [" ".join(c.split()[:260]) for c in chunks]

all_chunks = []
for f in sorted(glob.glob(f"{CORPUS_DIR}/*.txt")):
    for c in make_chunks(open(f, encoding="utf-8").read()):
        alpha = sum(ch.isalpha() or ch.isspace() for ch in c) / max(1, len(c))
        if alpha >= 0.7:
            all_chunks.append({"source": os.path.basename(f), "chunk": c})
N_CHUNKS = 300
selected = random.Random(SEED).sample(all_chunks, min(N_CHUNKS, len(all_chunks)))
print(f"candidate chunks: {len(all_chunks)} | selected: {len(selected)}")

In [ ]:
# Synthetic generation (Qwen2.5-7B-Instruct in 4-bit NF4 -> ~5.5 GB, fits the T4). Results are cached so a crash never costs the run.
from tqdm.auto import tqdm
RAW_CACHE = "raw_generations.json"
GEN_ID = "Qwen/Qwen2.5-7B-Instruct"

if os.path.exists(RAW_CACHE) and len(json.load(open(RAW_CACHE))["outputs"]) == len(selected):
    cache = json.load(open(RAW_CACHE)); GEN_ID = cache["generator"]
    raw_outputs = [(selected[i], o) for i, o in enumerate(cache["outputs"])]
    print("Loaded cached generations from", RAW_CACHE, "| generator:", GEN_ID)
else:
    safe_del("base_model", "gen_model", "trainer", "qmodel")      # make sure the GPU is empty
    bnb_gen = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
    try:
        gen_tok = AutoTokenizer.from_pretrained(GEN_ID)
        gen_model = AutoModelForCausalLM.from_pretrained(GEN_ID, quantization_config=bnb_gen,
                                                         torch_dtype=DTYPE, device_map={"": 0}).eval()
    except Exception as e:
        print("Could not load", GEN_ID, "->", repr(e)[:200], "\nFalling back to", MODEL_ID)
        GEN_ID = MODEL_ID
        gen_tok = AutoTokenizer.from_pretrained(GEN_ID)
        gen_model = AutoModelForCausalLM.from_pretrained(GEN_ID, torch_dtype=DTYPE, device_map={"": 0}).eval()
    gen_tok.padding_side = "left"
    if gen_tok.pad_token is None: gen_tok.pad_token = gen_tok.eos_token
    print("Generator model:", GEN_ID, "| GPU memory (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

    def gen_texts(texts, max_new=320):
        """Generate for a list of prompts; on CUDA OOM split the batch in two and retry."""
        try:
            enc = gen_tok(texts, return_tensors="pt", padding=True).to(gen_model.device)
            with torch.no_grad():
                out = gen_model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                         temperature=None, top_p=None, top_k=None,   # silences the sampling UserWarnings
                                         pad_token_id=gen_tok.pad_token_id)
            res = [gen_tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]
            del enc, out; torch.cuda.empty_cache()
            return res
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(texts) == 1: raise
            h = len(texts) // 2
            return gen_texts(texts[:h], max_new) + gen_texts(texts[h:], max_new)

    raw_outputs, BS = [], 16
    t0 = time.time()
    for i in tqdm(range(0, len(selected), BS)):
        batch = selected[i:i+BS]
        texts = [gen_tok.apply_chat_template([{"role": "user", "content": PROMPT_TEMPLATE.format(chunk=b["chunk"])}],
                                             tokenize=False, add_generation_prompt=True) for b in batch]
        for b, o in zip(batch, gen_texts(texts)):
            raw_outputs.append((b, o))
    print(f"generation time: {(time.time()-t0)/60:.1f} min")
    json.dump({"generator": GEN_ID, "outputs": [o for _, o in raw_outputs]}, open(RAW_CACHE, "w"))
    safe_del("gen_model", "gen_tok")
print("raw generations:", len(raw_outputs))

In [ ]:
def parse_pairs(raw):
    try:
        s, e = raw.index("["), raw.rindex("]") + 1
        data = json.loads(raw[s:e])
        return [d for d in data if isinstance(d, dict)]
    except Exception:
        out = []
        for m in re.finditer(r'\{[^{}]*"instruction"[^{}]*"response"[^{}]*\}', raw, re.S):
            try: out.append(json.loads(m.group(0)))
            except Exception: pass
        return out

BAD_REF = re.compile(r"\b(the text|the passage|this section|the document|the above|provided text)\b", re.I)
def content_words(t): return {w for w in re.findall(r"[a-z]+", t.lower()) if len(w) > 3}

rows, seen_q, rejected = [], set(), {"parse/format": 0, "reference": 0, "length": 0, "ungrounded": 0, "duplicate": 0}
n_raw_pairs = 0
for b, raw in raw_outputs:
    pairs = parse_pairs(raw)
    n_raw_pairs += len(pairs)
    if not pairs: rejected["parse/format"] += 1
    src_words = content_words(b["chunk"])
    for d in pairs:
        ins, res = d.get("instruction"), d.get("response")
        if not isinstance(ins, str) or not isinstance(res, str): rejected["parse/format"] += 1; continue
        ins, res = ins.strip(), res.strip()
        if BAD_REF.search(ins) or BAD_REF.search(res): rejected["reference"] += 1; continue
        if len(ins.split()) < 5 or not (8 <= len(res.split()) <= 120): rejected["length"] += 1; continue
        rw = content_words(res)
        if not rw or len(rw & src_words) / len(rw) < 0.70: rejected["ungrounded"] += 1; continue
        if ins.lower() in seen_q: rejected["duplicate"] += 1; continue
        seen_q.add(ins.lower())
        rows.append({"instruction": ins, "response": res, "source": b["source"]})
print(f"parsed pairs: {n_raw_pairs} | kept examples: {len(rows)} | rejected: {rejected}")
assert len(rows) >= 200, "too few examples - raise N_CHUNKS in the chunking cell, delete raw_generations.json and re-run"

with open("instruction_dataset.jsonl", "w", encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

tok_lens = [len(tokenizer(build_sft_text(r["instruction"], r["response"]))["input_ids"]) for r in rows]
print(f"avg tokens/example: {np.mean(tok_lens):.1f} | max: {max(tok_lens)} | total tokens: {sum(tok_lens):,}")
print("\n5 sample instruction-response pairs:")
for r in random.Random(SEED).sample(rows, 5):
    print("-" * 90, "\nQ:", r["instruction"], "\nA:", r["response"], f"\n[source: {r['source']}]")

In [ ]:
from datasets import Dataset
ds = Dataset.from_list(rows).train_test_split(test_size=0.20, seed=SEED)
train_ds, eval_ds = ds["train"], ds["test"]
print(f"Total: {len(rows)} | Train (80%): {len(train_ds)} | Eval (20%): {len(eval_ds)} | seed={SEED}")
train_ds = train_ds.map(lambda r: {"text": build_sft_text(r["instruction"], r["response"])})
eval_ds = eval_ds.map(lambda r: {"text": build_sft_text(r["instruction"], r["response"])})
pd.DataFrame(train_ds.remove_columns("text")).to_json("train_split.jsonl", orient="records", lines=True)
pd.DataFrame(eval_ds.remove_columns("text")).to_json("eval_split.jsonl", orient="records", lines=True)
print(f"\nInference: {len(rows)} grounded pairs averaging {np.mean(tok_lens):.0f} tokens ({sum(tok_lens):,} tokens in total) were kept out of {n_raw_pairs} generated; "
      f"the filters rejected {sum(rejected.values())} items (largest cause: '{max(rejected, key=rejected.get)}'). "
      "The 80/20 split uses a fixed seed so train/eval membership is reproducible.")

## Part B2 — QLoRA fine-tuning with **Adapter B (Balanced)**  [2 marks]
Exactly the assignment's Adapter B: **r = 16, alpha = 32, target modules = `q_proj`, `v_proj`**, dropout 0.05. The base model is loaded in 4-bit NF4 (double quantisation; compute dtype = the working dtype). Loss is computed on the **assistant response only** (`DataCollatorForCompletionOnlyLM`), using the same ChatML format as inference.

| Hyperparameter | Value |
|---|---|
| Per-device batch size | 8 |
| Gradient accumulation | 2 (effective batch 16) |
| Learning rate / schedule | 2e-4, cosine, 5 % warm-up |
| Epochs | 3 |
| Max sequence length | 512 |
| Precision | mixed precision (bf16 on Ampere+, fp16 on T4), gradient checkpointing |
| Seed | 42 |

In [ ]:
safe_del("base_model", "trainer", "qmodel", "ft_model", "base_bf16")
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
qmodel = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb4,
                                              torch_dtype=DTYPE, device_map={"": 0})
qmodel.config.use_cache = False
qmodel = prepare_model_for_kbit_training(qmodel, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                      lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")

# sanity check: the collator can only mask the prompt if the response template appears in the tokenised example
response_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
ex_ids = tokenizer(train_ds[0]["text"], add_special_tokens=False)["input_ids"]
assert any(ex_ids[i:i+len(response_ids)] == response_ids for i in range(len(ex_ids))), "response template not found in tokenised text"
collator = DataCollatorForCompletionOnlyLM(response_template=response_ids, tokenizer=tokenizer)

sft_args = SFTConfig(
    output_dir="hr_qlora_model", per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, learning_rate=2e-4, num_train_epochs=3,
    lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=5,
    eval_strategy="epoch", save_strategy="no", bf16=USE_BF16_TRAIN, fp16=not USE_BF16_TRAIN,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=512, dataset_text_field="text", report_to="none", seed=SEED)

trainer = SFTTrainer(model=qmodel, args=sft_args, train_dataset=train_ds, eval_dataset=eval_ds,
                     tokenizer=tokenizer, peft_config=lora_cfg, data_collator=collator)
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
expected_steps = math.ceil(len(train_ds) / (8 * 2)) * 3
print("expected optimizer steps:", expected_steps)

In [ ]:
t0 = time.time()
train_result = trainer.train()
print(f"training time: {(time.time()-t0)/60:.1f} min | global steps: {trainer.state.global_step} | peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")
assert trainer.state.global_step >= expected_steps * 0.9, "training stopped early - check the run"

log = pd.DataFrame(trainer.state.log_history)
display(log[[c for c in ["epoch", "step", "loss", "eval_loss"] if c in log.columns]].dropna(how="all", subset=[c for c in ["loss", "eval_loss"] if c in log.columns]))
trainer.model.save_pretrained("hr_adapter"); tokenizer.save_pretrained("hr_adapter")
print("adapter saved:", os.listdir("hr_adapter"))

In [ ]:
# Training-vs-eval loss summary + computed inference
tr = log.dropna(subset=["loss"]); ev = log.dropna(subset=["eval_loss"])
first_loss, last_loss = tr["loss"].iloc[0], tr["loss"].iloc[-1]
ev_list = [round(x, 3) for x in ev["eval_loss"]]
print(f"train loss: {first_loss:.3f} -> {last_loss:.3f} | eval loss per epoch: {ev_list}")
trend = "kept falling every epoch, so the adapter generalises to held-out pairs" if all(a > b for a, b in zip(ev_list, ev_list[1:])) \
        else "stopped improving / rose in a later epoch, a sign of mild over-fitting to the 3-epoch schedule"
print(f"\nInference (computed): Adapter B trains only {100*trainable/total:.2f}% of the parameters ({trainable:,}) on a frozen 4-bit base, which is why it fits comfortably on a T4. "
      f"Training loss fell from {first_loss:.3f} to {last_loss:.3f} and the eval loss {trend}. "
      f"The final train/eval gap is {last_loss - ev_list[-1]:+.3f}. Because loss is computed on the answer tokens only, capacity is spent on learning the policy answers, not on memorising the questions.")

## Part B3 — Baseline vs adapter (same 3 prompts, same greedy decoding)  [1 mark]
The adapter was trained against the 4-bit base and is applied here on top of the 16-bit base (the standard QLoRA inference setup).

In [ ]:
safe_del("trainer", "qmodel")
from peft import PeftModel
base_bf16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_bf16, "hr_adapter").eval()

adapter_outputs = []
for p in PROMPTS:
    text, dt, n = generate(ft_model, p, GREEDY, max_new_tokens=150)
    adapter_outputs.append({"prompt": p, "response": text})

baseline_df = pd.read_csv("baseline_outputs.csv")      # the Part A baseline, NOT regenerated
comparison = pd.DataFrame({"Prompt": PROMPTS,
                           "Baseline": baseline_df["response"],
                           "Fine_Tuned": [a["response"] for a in adapter_outputs]})
comparison.to_csv("baseline_vs_finetuned.csv", index=False)
for _, r in comparison.iterrows():
    print("=" * 100, "\nPROMPT:", r.Prompt, "\n\n--- BASELINE ---\n", r.Baseline, "\n\n--- FINE-TUNED ---\n", r.Fine_Tuned, "\n")

In [ ]:
rows_m = []
for _, r in comparison.iterrows():
    for label in ["Baseline", "Fine_Tuned"]:
        rows_m.append({"Prompt": r.Prompt[:45] + "...", "Model": label, **metrics(r[label])})
mdf = pd.DataFrame(rows_m); display(mdf)
agg = mdf.groupby("Model")[["words", "corpus_trigram_overlap", "distinct_2"]].mean().round(3)
agg["artifacts"] = mdf.groupby("Model")["artifact"].sum()
display(agg)
b, f_ = agg.loc["Baseline"], agg.loc["Fine_Tuned"]

verdict = []
verdict.append(f"Domain terminology/grounding: corpus-trigram overlap {b.corpus_trigram_overlap} -> {f_.corpus_trigram_overlap} "
               f"({'the adapter now uses more of the policy documents' + chr(39) + ' own wording' if f_.corpus_trigram_overlap > b.corpus_trigram_overlap else 'no gain in policy wording'}).")
verdict.append(f"Completeness/length: {b.words:.0f} -> {f_.words:.0f} words on average "
               f"({'shorter, more direct answers in the style of the training pairs' if f_.words < b.words else 'longer answers'}).")
verdict.append(f"Repetition (distinct-2, higher is better): {b.distinct_2} -> {f_.distinct_2}.")
verdict.append(f"Off-task artefacts: baseline {int(b.artifacts)} vs fine-tuned {int(f_.artifacts)}"
               f"{' - none remain' if int(f_.artifacts) == 0 else ' - some remain and count as a degradation'}.")
verdict.append("Hallucination risk: the adapter learned the style and vocabulary of the grounded Q&A, but a 1.7B model trained on a few hundred short pairs can still state "
               "specific figures that are not in the PDFs; any such figure must be checked against the source handbooks (read the side-by-side text above).")
print("Inference - Part B3 (computed + reading guide):")
for v in verdict: print(" *", v)

### Inference — Part B3 (what to check when you read the table above)
1. **Domain terminology / grounding** — `corpus_trigram_overlap` is the share of an answer's 3-word phrases that appear verbatim in the HR corpus; an increase means policy wording is being used.
2. **Correctness / hallucination** — compare each fine-tuned answer with the source PDFs. Invented numbers (e.g. a made-up number of leave days) are hallucinations even when the answer sounds confident.
3. **Completeness** — does the answer cover *every* part of the question (e.g. leave types **and** how applications are handled)?
4. **Degradations** — the `artifact` flag counts off-task continuations; with response-only loss and an `<|im_end|>` stop token these should vanish. If any remain, say so explicitly.

**Final verdict (write 4–5 lines after reading the outputs):** *what improved, what degraded, whether any hallucination remains.*

# PART C — Inference Optimization & Production Metrics  [8 marks]
All of Part C uses the **same base model as Part A** (`SmolLM2-1.7B-Instruct`, working dtype printed in the setup cell), the same fixed prompts, `max_new_tokens=150`, and the same seed, so differences come only from inference settings.

## Step 1 — Decoding strategies & sampling  [3 marks]
In HF `generate`, `top_k` defaults to 50 whenever sampling is on, which silently contaminates "pure" temperature and top-p runs. Here `top_k=0` / `top_p=1.0` are set explicitly wherever a filter must be off. Timing uses `perf_counter` with CUDA sync after one warm-up call, and speed is **actual new tokens / seconds**. Three temperatures (0.3, 0.7, 1.2) are compared.

In [ ]:
safe_del("ft_model", "base_bf16", "base_model")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0}).eval()

strategies = {
    "Greedy":               dict(do_sample=False),
    "Beam Search (4)":      dict(do_sample=False, num_beams=4),
    "Top-K (50, T=0.7)":    dict(do_sample=True, top_k=50, top_p=1.0, temperature=0.7),
    "Top-P (0.9, T=0.7)":   dict(do_sample=True, top_k=0, top_p=0.9, temperature=0.7),
    "Temp 0.3":             dict(do_sample=True, top_k=0, top_p=1.0, temperature=0.3),
    "Temp 0.7":             dict(do_sample=True, top_k=0, top_p=1.0, temperature=0.7),
    "Temp 1.2":             dict(do_sample=True, top_k=0, top_p=1.0, temperature=1.2),
}
generate(base_model, PROMPTS[0], GREEDY, max_new_tokens=20)   # warm-up

c1 = []
for name, cfg in strategies.items():
    for p in PROMPTS:
        text, dt, n = generate(base_model, p, cfg, max_new_tokens=150)
        c1.append({"Prompt": p, "Strategy": name, "Time_s": round(dt, 3), "New_tokens": n,
                   "Tok_per_s": round(n / dt, 1), "Response": text, **metrics(text)})
c1_df = pd.DataFrame(c1)
c1_df.to_csv("Part_C1_Decoding_Results.csv", index=False)
print("rows:", len(c1_df), "(7 strategies x 3 prompts)")

In [ ]:
# Required comparison table: prompts x strategies (text truncated for display; full text in the CSV)
pd.set_option("display.max_colwidth", 220)
order = list(strategies)
table = c1_df.pivot(index="Prompt", columns="Strategy", values="Response")[order]
table = table.loc[PROMPTS]; table.index = [f"Domain Prompt {i+1}" for i in range(3)]
display(table.apply(lambda col: col.map(lambda s: s[:220] + ("..." if len(s) > 220 else ""))))

print("\nSpeed / quality summary (mean over the 3 prompts):")
summary = c1_df.groupby("Strategy")[["Time_s", "New_tokens", "Tok_per_s", "corpus_trigram_overlap", "distinct_2"]].mean().round(3).loc[order]
summary["artifacts"] = c1_df.groupby("Strategy")["artifact"].sum().loc[order]
display(summary)

In [ ]:
# ~100-word analysis, assembled from the measured numbers above
S = summary
analysis = (f"I would deploy Top-P (p=0.9, T=0.7) for this open-ended HR assistant. It took {S.loc['Top-P (0.9, T=0.7)','Time_s']:.1f}s per answer versus "
            f"{S.loc['Greedy','Time_s']:.1f}s for greedy, while beam search needed {S.loc['Beam Search (4)','Time_s']:.1f}s, so diversity costs little speed. "
            f"Its distinct-2 was {S.loc['Top-P (0.9, T=0.7)','distinct_2']} against {S.loc['Greedy','distinct_2']} for greedy (less repetition), and its corpus-trigram overlap "
            f"was {S.loc['Top-P (0.9, T=0.7)','corpus_trigram_overlap']} versus {S.loc['Temp 1.2','corpus_trigram_overlap']} at T=1.2, where wording drifts from policy language. "
            f"Nucleus sampling trims unlikely tokens, keeping answers varied but on-policy.")
print(analysis); print("\nword count:", len(analysis.split()))
print("(The pick follows the domain tip: open-ended HR Q&A -> Top-P / moderate temperature. If the measured table contradicts a sentence above, edit that sentence.)")
BEST_NAME = "Top-P (0.9, T=0.7)"; BEST_CFG = strategies[BEST_NAME]

## Step 2 — Speculative decoding  [2 marks]
Target: `SmolLM2-1.7B-Instruct`. Draft: `SmolLM2-135M-Instruct` (same tokenizer/vocabulary; both fit in T4 VRAM together). The **identical formatted prompts as Step 1** are used with **greedy decoding**, so speculative decoding must return the same text as plain greedy — a built-in quality check. Every call is warmed up, repeated 3 times and averaged; speed = actual generated tokens / second.

In [ ]:
draft_model = AutoModelForCausalLM.from_pretrained(DRAFT_ID, torch_dtype=DTYPE, device_map={"": 0}).eval()
generate(base_model, PROMPTS[0], GREEDY, 20, assistant_model=draft_model)    # warm-up

REPEATS = 3
c2 = []
for p in PROMPTS:
    for mode, draft in [("Greedy (no draft)", None), ("Greedy + speculative", draft_model)]:
        times, toks = [], []
        for _ in range(REPEATS):
            text, dt, n = generate(base_model, p, GREEDY, 150, assistant_model=draft)
            times.append(dt); toks.append(n)
        c2.append({"Prompt": p[:50] + "...", "Mode": mode, "Time_s": round(np.mean(times), 3), "New_tokens": toks[0],
                   "Tok_per_s": round(np.mean(toks) / np.mean(times), 1), "Response": text})
c2_df = pd.DataFrame(c2); display(c2_df.drop(columns="Response"))

plain = c2_df[c2_df.Mode == "Greedy (no draft)"].reset_index(drop=True)
spec = c2_df[c2_df.Mode == "Greedy + speculative"].reset_index(drop=True)
speedup = spec.Tok_per_s.mean() / plain.Tok_per_s.mean()
identical = [a == b for a, b in zip(plain.Response, spec.Response)]
print(f"mean throughput: plain {plain.Tok_per_s.mean():.1f} tok/s | speculative {spec.Tok_per_s.mean():.1f} tok/s | speedup x{speedup:.2f}")
print("outputs identical to plain greedy:", identical)
c2_df.to_csv("Part_C2_Speculative_Results.csv", index=False)

In [ ]:
verdict = "faster" if speedup > 1.05 else ("slower" if speedup < 0.95 else "about the same speed")
lat_change = 100 * (1 - spec.Time_s.mean() / plain.Time_s.mean())
print(f"Inference (computed): speculative decoding was {verdict} than plain greedy (x{speedup:.2f} throughput, latency per 150-token answer changed by {lat_change:+.1f}%). "
      f"{sum(identical)}/{len(identical)} outputs were identical to plain greedy, "
      f"{'so there is no quality change, as expected for lossless greedy speculation' if all(identical) else 'the small differences come from 16-bit numerical non-determinism between verification batches, not from the method'}.")
print("Why: the 135M draft proposes several tokens and the 1.7B target verifies them in ONE forward pass. It pays off when (a) the target's forward pass is expensive and "
      "(b) the draft's tokens are accepted often. A 1.7B model is small, and in HF's implementation the draft is run token-by-token in Python, so its overhead can cancel "
      "the gain; the larger the target (7-8B), the more speculative decoding helps.")

## Step 3 — 4-bit quantization & production cost  [3 marks]
Each configuration is benchmarked in a **fresh load** (previous models deleted, peak-memory counter reset after loading so the number covers weights + generation), on the **same 10 domain prompts**, with the **best strategy from Step 1** (`Top-P 0.9, T=0.7`). Throughput = total generated tokens ÷ total seconds. **Cost / 1M tokens = (1,000,000 ÷ tok/s) ÷ 3600 × hourly rate** (₹3.50/hr T4, ₹12.00/hr A100). 4-bit config: `load_in_4bit=True, bnb_4bit_quant_type="nf4"`, compute dtype = working dtype.

In [ ]:
PROMPTS10 = PROMPTS + [
    "How many days of notice must an employee give to resign, and what does the exit process involve?",
    "What is the procedure for reporting workplace harassment or misconduct?",
    "Explain the performance appraisal process and how it affects promotions.",
    "What rules apply to code of conduct, conflict of interest and acceptance of gifts?",
    "Describe the policy on working hours, overtime and attendance.",
    "What benefits and allowances are employees eligible for, such as travel or medical?",
    "What disciplinary actions can the organisation take and what is the due process?",
]
assert len(PROMPTS10) == 10
safe_del("base_model", "draft_model")

def bench(name, quantize, use_spec):
    free()
    kw = dict(torch_dtype=DTYPE, device_map={"": 0})
    if quantize:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=DTYPE)
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw).eval()
    d = AutoModelForCausalLM.from_pretrained(DRAFT_ID, torch_dtype=DTYPE, device_map={"": 0}).eval() if use_spec else None
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()      # peak = loaded weights + generation
    generate(m, PROMPTS10[0], BEST_CFG, 20, assistant_model=d)           # warm-up
    tot_tok, tot_t, texts = 0, 0.0, []
    for p in PROMPTS10:
        text, dt, n = generate(m, p, BEST_CFG, 150, assistant_model=d)
        tot_tok += n; tot_t += dt; texts.append(text)
    peak = torch.cuda.max_memory_allocated() / 1024**3
    thr = tot_tok / tot_t
    cost = (1_000_000 / thr) / 3600 * HOURLY_RATE
    q = pd.DataFrame([metrics(t) for t in texts]).mean(numeric_only=True)
    del m, d; free()
    return {"Configuration": name, "Peak VRAM (GB)": round(peak, 2), "Throughput (tok/s)": round(thr, 1),
            "Cost / 1M tokens (Rs)": round(cost, 2), "Tokens generated": tot_tok,
            "corpus_trigram_overlap": round(q.corpus_trigram_overlap, 3), "distinct_2": round(q.distinct_2, 3)}

results = [bench(f"{DTYPE_NAME} (Part A baseline)", False, False),
           bench("4-bit NF4 quantized", True, False),
           bench("4-bit + Speculative Decoding", True, True)]
c3_df = pd.DataFrame(results); display(c3_df)
print(f"GPU: {GPU_NAME} | hourly rate: Rs {HOURLY_RATE} | decoding: {BEST_NAME} | prompts: {len(PROMPTS10)}")
c3_df.to_csv("Part_C3_Cost_Table.csv", index=False)

In [ ]:
# Deployment recommendation chosen from the measured table (cheapest config whose quality proxy is within 10% of the 16-bit baseline)
ref = c3_df.iloc[0]
ok = c3_df[c3_df.corpus_trigram_overlap >= 0.9 * ref.corpus_trigram_overlap]
best = ok.loc[ok["Cost / 1M tokens (Rs)"].idxmin()]
bf, q4, qs = c3_df.iloc[0], c3_df.iloc[1], c3_df.iloc[2]
print(f"Recommendation: deploy {best['Configuration']}. 4-bit changed peak VRAM from {bf['Peak VRAM (GB)']} GB to {q4['Peak VRAM (GB)']} GB and throughput "
      f"from {bf['Throughput (tok/s)']} to {q4['Throughput (tok/s)']} tok/s (cost Rs {bf['Cost / 1M tokens (Rs)']} -> Rs {q4['Cost / 1M tokens (Rs)']} per 1M tokens); "
      f"adding speculative decoding gives {qs['Throughput (tok/s)']} tok/s at Rs {qs['Cost / 1M tokens (Rs)']}. "
      f"The quality proxy (corpus-trigram overlap) was {bf['corpus_trigram_overlap']} / {q4['corpus_trigram_overlap']} / {qs['corpus_trigram_overlap']} for the three configurations, "
      f"so the pick is the cheapest configuration within 10% of 16-bit quality.")
print("\nNote: 4-bit mainly saves memory; bitsandbytes de-quantises weights on the fly, so on a small 1.7B model it is often not faster (and can be slower) than 16-bit - the measured table decides, not assumption. "
      "Its benefit is fitting a larger model (or more concurrent requests) in the same VRAM.")

## Optional extension (no marks) — fine-tuned adapter + 4-bit + speculative decoding
The realistic production stack. Skipped automatically if anything fails.

In [ ]:
try:
    safe_del("base_model", "draft_model")
    q_ext = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                 bnb_4bit_compute_dtype=DTYPE), torch_dtype=DTYPE, device_map={"": 0})
    q_ext = PeftModel.from_pretrained(q_ext, "hr_adapter").eval()
    d_ext = AutoModelForCausalLM.from_pretrained(DRAFT_ID, torch_dtype=DTYPE, device_map={"": 0}).eval()
    torch.cuda.reset_peak_memory_stats(); generate(q_ext, PROMPTS[0], GREEDY, 20, assistant_model=d_ext)
    tt, tn = 0.0, 0
    for p in PROMPTS:
        text, dt, n = generate(q_ext, p, GREEDY, 150, assistant_model=d_ext); tt += dt; tn += n
    thr = tn / tt
    print(f"Adapter + 4-bit + speculative: {thr:.1f} tok/s | peak {torch.cuda.max_memory_allocated()/1024**3:.2f} GB | Rs {(1e6/thr)/3600*HOURLY_RATE:.2f} / 1M tokens")
    del q_ext, d_ext; free()
except Exception as e:
    print("optional extension skipped:", repr(e)[:200])

# Final consistency + submission package
Everything below is generated from the variables above, so `assignment_summary.csv` always matches the notebook.

In [ ]:
summary_rows = [["Corpus files (domain_corpus/*.txt)", len(glob.glob(f"{CORPUS_DIR}/*.txt"))],
                ["Pages after cleaning", int(stats_df.Pages.iloc[-1])],
                ["Instruction examples", len(rows)], ["Training samples", len(train_ds)], ["Evaluation samples", len(eval_ds)],
                ["Adapter", "B: r=16, alpha=32, q_proj+v_proj"], ["GPU", GPU_NAME], ["Working dtype", DTYPE_NAME], ["Hourly rate (Rs)", HOURLY_RATE]]
for _, r in c3_df.iterrows():
    summary_rows += [[f"{r['Configuration']} - Peak VRAM (GB)", r["Peak VRAM (GB)"]],
                     [f"{r['Configuration']} - Throughput (tok/s)", r["Throughput (tok/s)"]],
                     [f"{r['Configuration']} - Cost/1M tokens (Rs)", r["Cost / 1M tokens (Rs)"]]]
pd.DataFrame(summary_rows, columns=["Metric", "Value"]).to_csv("assignment_summary.csv", index=False)
display(pd.DataFrame(summary_rows, columns=["Metric", "Value"]))

In [ ]:
# 1) File -> Save the notebook (with outputs) FIRST. 2) Set NOTEBOOK_PATH to THIS notebook, then run.
import subprocess, sys
NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/2025af05052_LLM_for_GenAI_1_fixed.ipynb"   # <- edit if different
SUB = "submission_1B"
shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbconvert"], check=True)
if os.path.exists(NOTEBOOK_PATH):
    shutil.copy2(NOTEBOOK_PATH, SUB)
    subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "html", NOTEBOOK_PATH, "--output-dir", SUB], check=True)
else:
    print("!! notebook not found at NOTEBOOK_PATH - fix the path and re-run, or File -> Download .ipynb and export HTML via File -> Print / nbconvert yourself")
shutil.copytree(CORPUS_DIR, f"{SUB}/domain_corpus")
for f in ["instruction_dataset.jsonl", "train_split.jsonl", "eval_split.jsonl", "baseline_outputs.csv", "baseline_vs_finetuned.csv",
          "Part_C1_Decoding_Results.csv", "Part_C2_Speculative_Results.csv", "Part_C3_Cost_Table.csv",
          "assignment_summary.csv", "corpus_statistics.csv"]:
    shutil.copy2(f, SUB)
shutil.make_archive("submission_1B", "zip", SUB)
print(subprocess.run(["unzip", "-l", "submission_1B.zip"], capture_output=True, text=True).stdout)
try:
    from google.colab import files; files.download("submission_1B.zip")
except Exception: pass